In [30]:
import pandas as pd
# importing the excel of each customers receipts to find each customer 
df1 = pd.read_excel("/Users/mohammad/Downloads/فاكتورهاي بهار 1403.xlsx")
df_x = pd.read_excel("/Users/mohammad/Downloads/اقلام فروش بهار 1403.xlsx")
customers_code = df_x["كد مشتري"].to_list()
customers_status = df_x["كد گروه مشتري"].to_list()
customer_status_dict = dict(zip(customers_code, customers_status))

# dict to add data in it
customer_dict = {}

# total profit 
total_profit = df1["سود و زيان"].to_list()[-1]

# deleting sum row
df = df1.iloc[:-1]

df = df.dropna()

df = df[df["نوع فروش"] == "قهوه و لوازم جانبي"]

# customer set
customer_list = set(df["نام مشتري"].to_list())

# itterate over each customer
for i in customer_list:
    customer_df = df[df["نام مشتري"]==i]
    customer_id = customer_df["كد مشتري"].to_list()[0]
    if customer_status_dict[customer_id] == 2:
        order_number = len(set(customer_df["شماره"].to_list()))
        customer_net =  sum(customer_df["خالص"].to_list())
        customer_profit = sum(customer_df["سود و زيان"].to_list())
        percent_of_total_profit = round(customer_profit/total_profit, 4)*100
        # adding these datas to dict
        customer_dict.update({i: {"id": customer_id, "number of orders": order_number, "Total Sale":customer_net, "Total Profit": customer_profit,
                            "Percent of Profit": percent_of_total_profit, "Profit to Sale Ratio": round(customer_profit/ customer_net,4)}})

    
sorted_customer_list = sorted(customer_dict.items(), key=lambda x: x[1]["Total Profit"], reverse=True)

# Convert the sorted list back to a dictionary
sorted_customer_dict = dict(sorted_customer_list)

df_final = pd.DataFrame.from_dict(sorted_customer_dict, orient='index')


# calculating Normalized PV 
max_pv = max(df_final["Total Sale"].to_list())
min_pv = min(df_final["Total Sale"].to_list())

df_final["Normalized_PV"] = ((df_final["Total Sale"] - min_pv) / (max_pv - min_pv))
df_final
# df_final.to_excel("2سود هر مشتری.xlsx")

,id,number of orders,Total Sale,Total Profit,Percent of Profit,Profit to Sale Ratio,Normalized_PV
كافه بيكري روستار سانا سنتر,21629.0,27,10308750000,5.828859e+09,8.50,0.5654,1.000000
كافه روستار آوا سنتر(شهرام رجبي),21432.0,18,5940550000,3.363142e+09,4.90,0.5661,0.576189
ويكافه (محمود فتاح),20973.0,12,4182050000,2.215525e+09,3.23,0.5298,0.405576
كافه بيكري رولن,23797.0,18,3079850000,1.867105e+09,2.72,0.6062,0.298638
كافه كي آر بي (علي كرباسچي ),20513.0,9,3226500000,1.812799e+09,2.64,0.5618,0.312867
...,...,...,...,...,...,...,...
محمدرضا رستمي ( كافه رز ),23590.0,1,4250000,1.477016e+06,0.00,0.3475,0.000238
كافه موچي(عليرضا بهادراني),24235.0,1,2100000,1.352798e+06,0.00,0.6442,0.000029
كافه نان زاگ(حميدرضا حاجيان),24143.0,1,3000000,1.282067e+06,0.00,0.4274,0.000116
كافه كاكتوس(پيمان حسين پور),24133.0,1,2500000,1.282067e+06,0.00,0.5128,0.000068


In [73]:
import pandas as pd
df = pd.read_excel("/Users/mohammad/Downloads/سود و زيان بهار 1403.xlsx")
# crating a dict of eah good and their profit percent
good_codes = df["كد كالا"].to_list()
good_profit_percent = df["حاشيه سود"].to_list()
good_profit_dict = dict(zip(good_codes,good_profit_percent))

# applying profit percent to ech receipt
df1 = pd.read_excel("/Users/mohammad/Downloads/اقلام فروش بهار 1403.xlsx")
df1["حاشيه سود"]= None
df1["سود"] = None

# selecting cafes
df1 = df1[df1["نوع فروش"] == "قهوه و لوازم جانبي"]
df1 = df1[df1["كد گروه مشتري"] == 2]


# adding percent of profit of each good and calculating the profit of each receipt
for i in df1.index.to_list():
    code = df1.loc[i,"كد كالا/خدمت"]
    profit_percent = good_profit_dict[code]
    df1.loc[i,"حاشيه سود"] = profit_percent
    df1.loc[i,"سود"] = df1.loc[i,"حاشيه سود"] * df1.loc[i,"خالص"]


cafe_data_dict = {}
cafe_list = set(df1["نام مشتري"].to_list())
for i in cafe_list :
    cafe_df = df1[df1["نام مشتري"] == i]
    cafe_df_goods = set(cafe_df['كالا/خدمت'].to_list())

    good_dict = {}
    for j in cafe_df_goods:
        data = cafe_df[cafe_df["كالا/خدمت"] == j]
        name = data["كالا/خدمت"].to_list()[0]
        number_of_orders = len(set(data["شماره"].to_list()))
        amount_of_sale = sum(data["خالص"].to_list())
        profit_line = data["حاشيه سود"].to_list()[0]
        total_profit = sum(data["سود"].to_list())

        good_dict.update({j: {"name": name,"number_of_orders" : number_of_orders, "amount_of_sale": amount_of_sale ,
                              "profit_line": profit_line, "total_profit": total_profit }})

    cafe_data_dict.update({i:good_dict})



# List to hold the data for the DataFrame
data = []

# Iterate through the dictionary
for customer, products in cafe_data_dict.items():
    for product_key, product_info in products.items():
        # Append a new row to the data list
        data.append([
            customer,
            product_info['name'],
            product_info['number_of_orders'],
            product_info['amount_of_sale'],
            product_info['profit_line'],
            product_info['total_profit']
        ])

# Create the DataFrame
df = pd.DataFrame(data, columns=['مشتری', 'محصول', 'تعداد فاکتور', 'خالص', 'حاشیه سود محصول', 'سود حاصل'])

# Group by 'مشتری' and calculate the sum of 'سود حاصل'
profit_sums = df.groupby('مشتری')['سود حاصل'].sum().reset_index()

# Rename the column to indicate it's the total profit
profit_sums.rename(columns={'سود حاصل': 'total_profit'}, inplace=True)

# Merge the total profit back to the original DataFrame
df = df.merge(profit_sums, on='مشتری')

# Sort the DataFrame based on 'total_profit'
df_sorted = df.sort_values(by='total_profit', ascending=False)

# Drop the total_profit column to return to original format
df_sorted = df_sorted.drop(columns='total_profit')

# Display the sorted DataFrame
# df_sorted.to_excel("سود هر مشتری به تفکیک.xlsx")

